# Skin Disease Detection — Google Colab Training
**Dataset:** HAM10000 (10,015 dermoscopy images, 7 classes)  
**Model:** ResNet50 fine-tuned (2-phase)  
**Drive structure:**
```
My Drive/medical_ai_data/skin/
  HAM10000_images_part_1/
  HAM10000_images_part_2/
  HAM10000_metadata.csv
```

In [ ]:
# ============================================================
# CELL 1 — Check GPU
# Runtime → Change runtime type → T4 GPU → Save
# ============================================================
import torch
print('GPU         :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU!')
print('CUDA        :', torch.version.cuda)
print('PyTorch     :', torch.__version__)
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM        : {vram:.1f} GB')

In [ ]:
# ============================================================
# CELL 2 — Install dependencies
# ============================================================
!pip install -q torch torchvision matplotlib seaborn scikit-learn pillow tqdm pandas opencv-python-headless albumentations

In [ ]:
# ============================================================
# CELL 3 — Mount Google Drive + verify
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

SKIN_BASE = '/content/drive/MyDrive/medical_ai_data/skin'
IMG_P1    = os.path.join(SKIN_BASE, 'HAM10000_images_part_1')
IMG_P2    = os.path.join(SKIN_BASE, 'HAM10000_images_part_2')
CSV_PATH  = os.path.join(SKIN_BASE, 'HAM10000_metadata.csv')

print('Checking Drive paths...')
for path, name in [(IMG_P1,'Part 1'),(IMG_P2,'Part 2'),(CSV_PATH,'CSV')]:
    if os.path.isdir(path):
        print(f'  {name}: ✅ {len(os.listdir(path))} files')
    elif os.path.isfile(path):
        print(f'  {name}: ✅ {os.path.getsize(path)//1024} KB')
    else:
        print(f'  {name}: ❌ NOT FOUND at {path}')

In [ ]:
# ============================================================
# CELL 4 — Copy images to Colab local storage (MUCH faster than reading from Drive)
# This copies all images to /content/skin_images/ once
# Training reads from local SSD not Drive → 10x faster per epoch
# ============================================================
import shutil
from tqdm import tqdm

LOCAL_IMG_DIR = '/content/skin_images'
os.makedirs(LOCAL_IMG_DIR, exist_ok=True)

existing = len(os.listdir(LOCAL_IMG_DIR))
if existing > 5000:
    print(f'Images already copied: {existing} files in {LOCAL_IMG_DIR}')
else:
    print('Copying images from Drive to local Colab storage...')
    print('(This takes 3-5 minutes but training will be 10x faster)')
    total = 0
    for src_folder in [IMG_P1, IMG_P2]:
        if not os.path.exists(src_folder):
            continue
        files = [f for f in os.listdir(src_folder)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for fname in tqdm(files, desc=f'  Copying {os.path.basename(src_folder)}'):
            src  = os.path.join(src_folder, fname)
            dest = os.path.join(LOCAL_IMG_DIR, fname)
            if not os.path.exists(dest):
                shutil.copy2(src, dest)
            total += 1
    print(f'\n✅ Copied {total} images to {LOCAL_IMG_DIR}')

In [ ]:
# ============================================================
# CELL 5 — Imports
# ============================================================
import sys, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.benchmark = True  # faster training on T4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('All imports ✅')

In [ ]:
# ============================================================
# CELL 6 — Configuration
# ============================================================
OUTPUT_DIR   = '/content/skin_output'
SAVE_DIR     = '/content/drive/MyDrive/medical_ai_models'
BEST_PATH    = f'{OUTPUT_DIR}/skin_model_best.pth'
FINAL_PATH   = f'{OUTPUT_DIR}/skin_model.pth'
META_PATH    = f'{OUTPUT_DIR}/skin_metadata.json'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,   exist_ok=True)

# Training config — tuned for T4 GPU
BATCH_SIZE  = 32
EPOCHS_P1   = 15   # frozen backbone
EPOCHS_P2   = 10   # full fine-tuning
LR_P1       = 3e-4 # higher than before — converges faster
LR_P2       = 1e-5 # lower — gentle fine-tuning
IMG_SIZE    = 224
NUM_WORKERS = 2

print('Configuration:')
print(f'  BATCH_SIZE : {BATCH_SIZE}')
print(f'  EPOCHS P1  : {EPOCHS_P1}')
print(f'  EPOCHS P2  : {EPOCHS_P2}')
print(f'  LR P1      : {LR_P1}')
print(f'  LR P2      : {LR_P2}')
print(f'  IMG SIZE   : {IMG_SIZE}')

In [ ]:
# ============================================================
# CELL 7 — Class definitions
# ============================================================
CLASS_NAMES = [
    'Melanoma',          # mel
    'Nevus',             # nv
    'Basal_Cell_Ca',     # bcc
    'Actinic_Keratosis', # akiec
    'Benign_Keratosis',  # bkl
    'Dermatofibroma',    # df
    'Vascular_Lesion'    # vasc
]

DX_MAP = {
    'mel'  : 'Melanoma',
    'nv'   : 'Nevus',
    'bcc'  : 'Basal_Cell_Ca',
    'akiec': 'Actinic_Keratosis',
    'bkl'  : 'Benign_Keratosis',
    'df'   : 'Dermatofibroma',
    'vasc' : 'Vascular_Lesion'
}

CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASS_NAMES)}

print(f'Classes ({len(CLASS_NAMES)}):')
for i, c in enumerate(CLASS_NAMES):
    print(f'  [{i}] {c}')

In [ ]:
# ============================================================
# CELL 8 — Load CSV + build image index from LOCAL storage
# ============================================================
print('Loading HAM10000 metadata...')
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().lower() for c in df.columns]

print(f'  CSV shape: {df.shape}')
print(f'  Columns  : {list(df.columns)}')

# Build index from LOCAL copy (fast reads)
print('\nBuilding image index from local storage...')
img_index = {}
for fname in os.listdir(LOCAL_IMG_DIR):
    if fname.lower().endswith(('.jpg','.jpeg','.png')):
        img_id = os.path.splitext(fname)[0]
        img_index[img_id] = os.path.join(LOCAL_IMG_DIR, fname)

print(f'  Total images indexed: {len(img_index)}')

df['path']       = df['image_id'].map(img_index)
df['label_name'] = df['dx'].map(DX_MAP)
df['label_idx']  = df['label_name'].map(CLASS_TO_IDX)

before = len(df)
df = df[df['path'].notna() & df['label_name'].notna()].reset_index(drop=True)
print(f'  Valid samples: {len(df)} (dropped {before - len(df)})')

print('\nClass distribution:')
for cls in CLASS_NAMES:
    n   = (df['label_name'] == cls).sum()
    pct = n / len(df) * 100
    print(f'  {cls:25s}: {n:5d} ({pct:5.1f}%)')

In [ ]:
# ============================================================
# CELL 9 — Train/Val split (no lesion ID leakage)
# ============================================================
print('Splitting dataset...')

if 'lesion_id' in df.columns:
    # Split by unique lesion — same lesion cant appear in train AND val
    unique_lesions = df['lesion_id'].unique()
    train_lesions, val_lesions = train_test_split(
        unique_lesions, test_size=0.15, random_state=42)
    train_df = df[df['lesion_id'].isin(train_lesions)].reset_index(drop=True)
    val_df   = df[df['lesion_id'].isin(val_lesions)].reset_index(drop=True)
    print('  Split by lesion_id (prevents data leakage) ✅')
else:
    train_df, val_df = train_test_split(
        df, test_size=0.15, stratify=df['label_idx'], random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'  Train: {len(train_df)} | Val: {len(val_df)}')

print('\nTrain class breakdown:')
for cls in CLASS_NAMES:
    n = (train_df['label_name'] == cls).sum()
    print(f'  {cls:25s}: {n}')

In [ ]:
# ============================================================
# CELL 10 — Dataset class
# ============================================================
class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row['label_idx'])
        try:
            img = Image.open(row['path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), 0)
        if self.transform:
            img = self.transform(img)
        return img, label

print('SkinDataset defined ✅')

In [ ]:
# ============================================================
# CELL 11 — Transforms
# Dermoscopy-specific augmentation:
# - Strong rotation (moles look same at any angle)
# - Color jitter (lighting varies between dermatoscopes)
# - NO WeightedRandomSampler (caused instability with small VRAM)
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),  # full rotation — moles are rotation invariant
    transforms.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.25,
        hue=0.05
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),  # slight position shift
        scale=(0.9, 1.1)       # slight zoom
    ),
    transforms.ToTensor(),
    transforms.Normalize([0.7630, 0.5456, 0.5700],   # HAM10000 specific mean
                         [0.1409, 0.1527, 0.1704])    # HAM10000 specific std
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.7630, 0.5456, 0.5700],
                         [0.1409, 0.1527, 0.1704])
])

train_ds = SkinDataset(train_df, transform=train_transform)
val_ds   = SkinDataset(val_df,   transform=val_transform)

train_loader = DataLoader(
    train_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = NUM_WORKERS,
    pin_memory  = True
)
val_loader = DataLoader(
    val_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True
)

print(f'Train loader: {len(train_loader)} batches')
print(f'Val loader  : {len(val_loader)} batches')
print('Using HAM10000-specific normalization (not ImageNet) ✅')

In [ ]:
# ============================================================
# CELL 12 — Build ResNet50 model
# FIXED: removed BatchNorm1d from head (caused val instability)
# Using clean simple head — more stable training
# ============================================================
print('Building ResNet50 model...')

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Phase 1: freeze all except layer3 + layer4 + fc
# Unfreezing layer3 too gives more expressiveness for skin patterns
for name, param in model.named_parameters():
    if 'layer3' not in name and 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

# Simple clean head — NO BatchNorm (caused instability)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.5),
    nn.Linear(512, len(CLASS_NAMES))
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')
print(f'Frozen params   : {total-trainable:,}')
print('Head: Linear(2048→512) → ReLU → Dropout(0.5) → Linear(512→7)')

In [ ]:
# ============================================================
# CELL 13 — Loss + Optimizer + Scheduler
# FIXED:
#   - Using ReduceLROnPlateau instead of CosineAnnealing
#     → adapts to val loss instead of running on fixed schedule
#   - Removed label_smoothing (was hurting small minority classes)
#   - Class weights handle imbalance in loss function
# ============================================================

# Class weights from TRAINING set only
label_counts  = train_df['label_idx'].value_counts().sort_index()
total_train   = len(train_df)
class_weights = torch.tensor(
    [total_train / (len(CLASS_NAMES) * label_counts[i])
     for i in range(len(CLASS_NAMES))],
    dtype=torch.float
).to(device)

print('Class weights (from train set):')
for cls, w in zip(CLASS_NAMES, class_weights.cpu().numpy()):
    print(f'  {cls:25s}: {w:.3f}')

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_P1, weight_decay=1e-3  # stronger regularization
)

# ReduceLROnPlateau: reduces LR when val_loss stops improving
# patience=3 means wait 3 epochs before reducing
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5,
    patience=3, min_lr=1e-7, verbose=True
)

print(f'\nCriterion : CrossEntropyLoss with class weights')
print(f'Optimizer : AdamW (lr={LR_P1}, weight_decay=1e-3)')
print(f'Scheduler : ReduceLROnPlateau (patience=3, factor=0.5)')

In [ ]:
# ============================================================
# CELL 14 — Train + Val + EarlyStopping
# ============================================================
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='  Train', leave=False, ncols=80):
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='  Val  ', leave=False, ncols=80):
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            out         = model(imgs)
            total_loss += criterion(out, labels).item()
            correct    += (out.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total


class EarlyStopping:
    """Stop training when val_loss stops improving."""
    def __init__(self, patience=5, min_delta=0.001):
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best_loss = float('inf')
        self.stop      = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

print('Train/val/EarlyStopping functions defined ✅')

In [ ]:
# ============================================================
# CELL 15 — PHASE 1 Training (layer3 + layer4 + fc, 15 epochs)
# ============================================================
import time

history  = {k: [] for k in ['tl','vl','ta','va']}
best_acc = 0.0
early_stop = EarlyStopping(patience=5)

print('=' * 55)
print('PHASE 1 — Training layer3 + layer4 + head')
print(f'Epochs: {EPOCHS_P1} | LR: {LR_P1}')
print('=' * 55)

for epoch in range(1, EPOCHS_P1 + 1):
    t0 = time.time()

    tl, ta = train_epoch(model, train_loader, criterion, optimizer)
    vl, va = val_epoch(model, val_loader, criterion)

    # ReduceLROnPlateau steps on val_loss
    scheduler.step(vl)
    early_stop(vl)

    history['tl'].append(tl); history['vl'].append(vl)
    history['ta'].append(ta); history['va'].append(va)

    saved = ''
    if va > best_acc:
        best_acc = va
        torch.save(model.state_dict(), BEST_PATH)
        saved = '  💾 best'

    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:02d}/{EPOCHS_P1} | '
          f'Train {ta*100:.2f}% loss {tl:.4f} | '
          f'Val {va*100:.2f}% loss {vl:.4f} | '
          f'LR {lr_now:.2e} | '
          f'{time.time()-t0:.0f}s{saved}')

    if early_stop.stop:
        print(f'  Early stopping at epoch {epoch} (val_loss not improving for {early_stop.patience} epochs)')
        break

print(f'\n✅ Phase 1 best val accuracy: {best_acc*100:.2f}%')

In [ ]:
# ============================================================
# CELL 16 — PHASE 2 Training (all layers, 10 epochs)
# ============================================================
print('Loading best phase 1 weights...')
model.load_state_dict(torch.load(BEST_PATH, map_location=device))

# Unfreeze ALL layers
for param in model.parameters():
    param.requires_grad = True

total_p2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'All {total_p2:,} params unfrozen')

# Different LRs per layer group — lower for early layers (already well-trained)
optimizer2 = optim.AdamW([
    {'params': model.layer1.parameters(), 'lr': LR_P2 * 0.1},
    {'params': model.layer2.parameters(), 'lr': LR_P2 * 0.1},
    {'params': model.layer3.parameters(), 'lr': LR_P2 * 0.5},
    {'params': model.layer4.parameters(), 'lr': LR_P2},
    {'params': model.fc.parameters(),     'lr': LR_P2},
], weight_decay=1e-3)

scheduler2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer2, mode='min', factor=0.5,
    patience=3, min_lr=1e-9, verbose=True
)

early_stop2 = EarlyStopping(patience=5)

print('\n' + '=' * 55)
print('PHASE 2 — Full fine-tuning with layer-wise LR')
print(f'Epochs: {EPOCHS_P2} | Base LR: {LR_P2}')
print('=' * 55)

for epoch in range(1, EPOCHS_P2 + 1):
    t0 = time.time()

    tl, ta = train_epoch(model, train_loader, criterion, optimizer2)
    vl, va = val_epoch(model, val_loader, criterion)

    scheduler2.step(vl)
    early_stop2(vl)

    history['tl'].append(tl); history['vl'].append(vl)
    history['ta'].append(ta); history['va'].append(va)

    saved = ''
    if va > best_acc:
        best_acc = va
        torch.save(model.state_dict(), BEST_PATH)
        saved = '  💾 best'

    lr_now = optimizer2.param_groups[-1]['lr']
    print(f'Epoch {epoch:02d}/{EPOCHS_P2} | '
          f'Train {ta*100:.2f}% loss {tl:.4f} | '
          f'Val {va*100:.2f}% loss {vl:.4f} | '
          f'LR {lr_now:.2e} | '
          f'{time.time()-t0:.0f}s{saved}')

    if early_stop2.stop:
        print(f'  Early stopping at epoch {epoch}')
        break

print(f'\n✅ Phase 2 complete | Final best: {best_acc*100:.2f}%')

In [ ]:
# ============================================================
# CELL 17 — Training curves
# ============================================================
total_epochs = len(history['ta'])
ep_range     = range(1, total_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Skin Disease Detection — Training Results', fontsize=14, fontweight='bold')

ax1.plot(ep_range, history['tl'], color='#534AB7', label='Train loss', linewidth=2)
ax1.plot(ep_range, history['vl'], color='#D85A30', label='Val loss',   linewidth=2)
if total_epochs > EPOCHS_P1:
    ax1.axvline(x=EPOCHS_P1+0.5, color='gray', linestyle='--', alpha=0.7, label='Phase 2')
ax1.set_title('Loss Curve'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep_range, [a*100 for a in history['ta']], color='#534AB7', label='Train acc', linewidth=2)
ax2.plot(ep_range, [a*100 for a in history['va']], color='#D85A30', label='Val acc',   linewidth=2)
if total_epochs > EPOCHS_P1:
    ax2.axvline(x=EPOCHS_P1+0.5, color='gray', linestyle='--', alpha=0.7, label='Phase 2')
ax2.set_title('Accuracy Curve'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/skin_training_curves.png', dpi=120)
plt.show()
print('Saved: skin_training_curves.png')

In [ ]:
# ============================================================
# CELL 18 — Confusion matrix + classification report
# ============================================================
model.load_state_dict(torch.load(BEST_PATH, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc='Evaluating'):
        out = model(imgs.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print('\nClassification Report:')
print('=' * 70)
print(classification_report(all_labels, all_preds,
                             target_names=CLASS_NAMES, digits=4))

cm  = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Confusion Matrix — Skin Disease', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.xticks(rotation=25, ha='right'); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/skin_confusion_matrix.png', dpi=120)
plt.show()
print('Saved: skin_confusion_matrix.png')

In [ ]:
# ============================================================
# CELL 19 — Grad-CAM visualization
# ============================================================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model       = model
        self.gradients   = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, m, i, o): self.activations = o.detach()
    def _save_gradient(self, m, gi, go): self.gradients   = go[0].detach()

    def generate(self, tensor, class_idx=None):
        self.model.eval()
        out = self.model(tensor)
        if class_idx is None: class_idx = out.argmax(1).item()
        self.model.zero_grad()
        out[0, class_idx].backward()
        weights = self.gradients.mean(dim=[2,3], keepdim=True)
        cam     = torch.relu((weights * self.activations).sum(1, keepdim=True))
        cam     = (cam - cam.min()) / (cam.max() + 1e-8)
        cam     = cv2.resize(cam.squeeze().cpu().numpy(), (IMG_SIZE, IMG_SIZE))
        heatmap = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET)
        probs   = out.softmax(1)[0].detach().cpu().numpy()
        return heatmap, class_idx, probs

gradcam = GradCAM(model, model.layer4[-1].conv3)

preprocess_gc = transforms.Compose([
    transforms.Resize((256,256)), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.7630,0.5456,0.5700],[0.1409,0.1527,0.1704])
])

fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(12, 4*len(CLASS_NAMES)))
fig.suptitle('Grad-CAM — Skin Lesion Explanations', fontsize=14, fontweight='bold')

for row, cls in enumerate(CLASS_NAMES):
    cls_df = val_df[val_df['label_name'] == cls]
    if len(cls_df) == 0:
        for col in range(3): axes[row][col].axis('off')
        continue
    sample  = cls_df.sample(1).iloc[0]
    img_pil = Image.open(sample['path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_np  = np.array(img_pil)
    tensor  = preprocess_gc(img_pil).unsqueeze(0).to(device)
    heatmap, pred_idx, probs = gradcam.generate(tensor)
    overlay = cv2.addWeighted(img_np, 0.55, cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB), 0.45, 0)
    pred_cls = CLASS_NAMES[pred_idx]
    color    = 'green' if pred_idx == CLASS_TO_IDX[cls] else 'red'

    axes[row][0].imshow(img_np)
    axes[row][0].set_title(f'Actual: {cls}', fontsize=9, fontweight='bold')
    axes[row][0].axis('off')
    axes[row][1].imshow(overlay)
    axes[row][1].set_title(f'Pred: {pred_cls} ({probs[pred_idx]*100:.1f}%)', fontsize=9, color=color)
    axes[row][1].axis('off')
    axes[row][2].barh(CLASS_NAMES, probs*100,
                       color=['green' if i==pred_idx else '#ddd' for i in range(len(CLASS_NAMES))])
    axes[row][2].set_xlim(0,100); axes[row][2].set_title('Confidence %', fontsize=8)
    axes[row][2].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/skin_gradcam.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: skin_gradcam.png')

In [ ]:
# ============================================================
# CELL 20 — Save model + metadata
# ============================================================
torch.save(model.state_dict(), FINAL_PATH)

metadata = {
    'model_name'       : 'ResNet50 Skin Disease Detector',
    'version'          : '2.0',
    'num_classes'      : len(CLASS_NAMES),
    'classes'          : CLASS_NAMES,
    'class_to_idx'     : CLASS_TO_IDX,
    'input_size'       : [IMG_SIZE, IMG_SIZE],
    'normalize_mean'   : [0.7630, 0.5456, 0.5700],
    'normalize_std'    : [0.1409, 0.1527, 0.1704],
    'best_val_accuracy': round(best_acc * 100, 2),
    'total_epochs'     : len(history['ta']),
    'architecture'     : 'ResNet50 2-phase fine-tune, layer-wise LR',
    'dataset'          : 'HAM10000',
    'scan_type'        : 'Dermoscopy Photo',
    'disease_info': {
        'Melanoma'          : {'description':'Most dangerous skin cancer from melanocytes.','symptoms':['Asymmetric mole','Irregular border','Multiple colors','Diameter >6mm','Evolving'],'treatment':'Surgery, immunotherapy, targeted therapy','risk':'Critical','emergency':False},
        'Nevus'             : {'description':'Common benign mole.','symptoms':['Round shape','Uniform color','Stable size'],'treatment':'Monitoring only','risk':'Low','emergency':False},
        'Basal_Cell_Ca'     : {'description':'Most common skin cancer, grows slowly.','symptoms':['Pearly bump','Flat scar','Bleeding sore'],'treatment':'Mohs surgery, excision','risk':'Moderate','emergency':False},
        'Actinic_Keratosis' : {'description':'Sun-damage precancerous patch.','symptoms':['Rough dry patch','Itching','Burning'],'treatment':'Cryotherapy, topical 5-FU','risk':'Moderate','emergency':False},
        'Benign_Keratosis'  : {'description':'Benign waxy seborrheic growth.','symptoms':['Waxy stuck-on look','Brown/black color'],'treatment':'No treatment needed','risk':'Low','emergency':False},
        'Dermatofibroma'    : {'description':'Benign fibrous nodule.','symptoms':['Small firm bump','Dimples when pinched'],'treatment':'No treatment needed','risk':'Very low','emergency':False},
        'Vascular_Lesion'   : {'description':'Abnormal blood vessel formations.','symptoms':['Red/purple color','Blanches under pressure'],'treatment':'Laser therapy','risk':'Low','emergency':False}
    }
}

with open(META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✅ skin_model.pth saved')
print(f'✅ skin_metadata.json saved')
print(f'✅ Best val accuracy: {best_acc*100:.2f}%')

In [ ]:
# ============================================================
# CELL 21 — Save everything to Google Drive
# ============================================================
os.makedirs(SAVE_DIR, exist_ok=True)

files_to_save = [
    f'{OUTPUT_DIR}/skin_model.pth',
    f'{OUTPUT_DIR}/skin_metadata.json',
    f'{OUTPUT_DIR}/skin_training_curves.png',
    f'{OUTPUT_DIR}/skin_confusion_matrix.png',
    f'{OUTPUT_DIR}/skin_gradcam.png',
]

for fp in files_to_save:
    if os.path.exists(fp):
        dest = os.path.join(SAVE_DIR, os.path.basename(fp))
        shutil.copy2(fp, dest)
        print(f'  Saved: {os.path.basename(fp)}')

print(f'\n✅ All files saved to Drive → medical_ai_models/')
print(f'\nDownload to your project:')
print(f'  skin_model.pth      → backend/model/')
print(f'  skin_metadata.json  → backend/model/')